# Análisis de Seguridad de Repositorios

Este notebook consolida y analiza los resultados de seguridad generados por herramientas como Syft (SBOM), Grype (Vulnerabilidades) y CodeQL (SAST).

In [ ]:
import pandas as pd
import json
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Cargar Datos de Vulnerabilidades (Grype)

In [ ]:
grype_files = glob.glob('../data/results/*_grype.json')
vuln_data = []

for file in grype_files:
    repo_name = os.path.basename(file).replace('_grype.json', '')
    try:
        with open(file, 'r', encoding='utf-8') as f:
            content = json.load(f)
            for match in content.get('matches', []):
                vuln = match.get('vulnerability', {})
                artifact = match.get('artifact', {})
                vuln_data.append({
                    'repo': repo_name,
                    'id': vuln.get('id', 'Desconocido'),
                    'severity': vuln.get('severity', 'Unknown'),
                    'package': artifact.get('name', 'N/A'),
                    'version': artifact.get('version', 'N/A')
                })
    except Exception as e:
        print(f"Error al leer el archivo {file}: {e}")

df_vulns = pd.DataFrame(vuln_data)
if not df_vulns.empty:
    display(df_vulns.head())
else:
    print("No se encontraron datos de vulnerabilidades.")

## 2. Análisis Cuantitativo de Vulnerabilidades (SCA)

In [ ]:
if not df_vulns.empty:
    # Total de vulnerabilidades por repo
    repo_counts = df_vulns['repo'].value_counts()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=repo_counts.values, y=repo_counts.index, hue=repo_counts.index, palette='viridis', legend=False)
    plt.title('Total de Vulnerabilidades por Repositorio')
    plt.xlabel('Cantidad de Vulnerabilidades')
    plt.ylabel('Repositorio')
    plt.show()
    
    # Severidad general
    severity_counts = df_vulns['severity'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(severity_counts, labels=severity_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title('Distribución de Severidad de las Vulnerabilidades (SCA)')
    plt.show()
else:
    print("No hay datos para graficar.")

## 3. Cargar Datos y Análisis SAST (CodeQL)

In [ ]:
codeql_files = glob.glob('../data/results/*_codeql.sarif')
sast_data = []

for file in codeql_files:
    repo_name = os.path.basename(file).replace('_codeql.sarif', '')
    try:
        with open(file, 'r', encoding='utf-8') as f:
            content = json.load(f)
            runs = content.get('runs', [])
            for run in runs:
                results = run.get('results', [])
                for result in results:
                    rule_id = result.get('ruleId', 'Desconocido')
                    sast_data.append({
                        'repo': repo_name,
                        'rule_id': rule_id
                    })
    except Exception as e:
        print(f"Error leyendo archivo SARIF de CodeQL: {e}")

df_sast = pd.DataFrame(sast_data)
if not df_sast.empty:
    display(df_sast.head())
    
    # Extraer el top de vulnerabilidades/CWE de SAST
    rule_counts = df_sast['rule_id'].value_counts().head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=rule_counts.values, y=rule_counts.index, hue=rule_counts.index, palette='magma', legend=False)
    plt.title('Top 10 Reglas / Tipos de Vulnerabilidades Detectadas (CodeQL / SAST)')
    plt.xlabel('Frecuencia')
    plt.ylabel('Regla (CWE / Descripción)')
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron resultados de análisis estático (CodeQL).")

## 4. Análisis de Licencias (desde SBOM generado por Syft)

In [ ]:
sbom_files = glob.glob('../data/results/*_sbom.json')
license_data = []

for file in sbom_files:
    repo_name = os.path.basename(file).replace('_sbom.json', '')
    try:
        with open(file, 'r', encoding='utf-8') as f:
            content = json.load(f)
            artifacts = content.get('artifacts', [])
            for artifact in artifacts:
                licenses = artifact.get('licenses', [])
                for lic in licenses:
                    lic_name = lic if isinstance(lic, str) else lic.get('value', 'Desconocida')
                    license_data.append({
                        'repo': repo_name,
                        'license': lic_name
                    })
    except Exception as e:
        print(f"Error leyendo SBOM para licencias: {e}")

df_licenses = pd.DataFrame(license_data)
if not df_licenses.empty:
    lic_counts = df_licenses['license'].value_counts().head(10)
    plt.figure(figsize=(8, 8))
    plt.pie(lic_counts, labels=lic_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
    plt.title('Top Licencias Usadas en Dependencias del Proyecto')
    plt.show()
else:
    print("No se encontraron datos de licencias en los SBOMs.")